In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OSQ_D.xpt"
csv_file = DATA_DIR / "OSQ_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched original XPT for fidelity checking
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 OSQ_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "OSQ010A",
    "OSQ010B",
    "OSQ010C",
    "OSQ020A",
    "OSQ020B",
    "OSQ020C",
    "OSD030AA",
    "OSQ040AA",
    "OSD050AA",
    "OSD030AB",
    "OSQ040AB",
    "OSD050AB",
    "OSD030AC",
    "OSQ040AC",
    "OSD050AC",
    "OSD030BA",
    "OSQ040BA",
    "OSD050BA",
    "OSD030BB",
    "OSQ040BB",
    "OSD050BB",
    "OSD030BC",
    "OSQ040BC",
    "OSD050BC",
    "OSD030BD",
    "OSQ040BD",
    "OSD050BD",
    "OSD030BE",
    "OSQ040BE",
    "OSD050BE",
    "OSD030BF",
    "OSQ040BF",
    "OSD050BF",
    "OSD030BG",
    "OSQ040BG",
    "OSD050BG",
    "OSD030CA",
    "OSQ040CA",
    "OSD050CA",
    "OSD030CB",
    "OSQ040CB",
    "OSD050CB",
    "OSD030CC",
    "OSQ040CC",
    "OSD050CC",
    "OSD030CD",
    "OSQ040CD",
    "OSD050CD",
    "OSD030CE",
    "OSQ040CE",
    "OSD050CE",
    "OSD030CF",
    "OSQ040CF",
    "OSD050CF",
    "OSD030CG",
    "OSQ040CG",
    "OSD030CH",
    "OSQ040CH",
    "OSD030CI",
    "OSQ040CI",
    "OSD030CJ",
    "OSQ040CJ",
    "OSQ080",
    "OSQ090A",
    "OSQ100A",
    "OSD110A",
    "OSQ120A",
    "OSQ090B",
    "OSQ100B",
    "OSD110B",
    "OSQ120B",
    "OSQ090C",
    "OSQ100C",
    "OSD110C",
    "OSQ120C",
    "OSQ090D",
    "OSQ100D",
    "OSD110D",
    "OSQ120D",
    "OSQ090E",
    "OSQ100E",
    "OSD110E",
    "OSQ120E",
    "OSQ090F",
    "OSQ100F",
    "OSD110F",
    "OSQ120F",
    "OSQ090G",
    "OSQ100G",
    "OSD110G",
    "OSQ120G",
    "OSQ090H",
    "OSQ100H",
    "OSD110H",
    "OSQ120H",
    "OSQ090I",
    "OSQ100I",
    "OSD110I",
    "OSQ120I",
    "OSQ090J",
    "OSQ100J",
    "OSD110J",
    "OSQ120J",
    "OSQ060",
    "OSQ070",
    "OSQ130",
    "OSQ140Q",
    "OSQ140U",
    "OSQ150",
    "OSQ160A",
    "OSQ160B",
    "OSQ170",
    "OSQ180",
    "OSQ190",
    "OSQ200",
    "OSQ210",
    "OSQ220"
]


if len(df) != 4979:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 118:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "OSQ_D columns/order differ.\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: OSQ_D structure = "
    "4,979 rows x 118 columns."
)


# ============================================================
# 4. KNOWN SAS/XPORT TINY-ZERO ARTIFACT CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- TINY VALUE CHECK ---")

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: OSQ_D unexpectedly contains "
        "tiny XPORT values. "
        "Do NOT automatically change them to zero."
    )


print(
    "PASS: OSQ_D contains no tiny-value artifacts."
)


# ============================================================
# 5. TRUE ZERO CHECK
# ============================================================

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


zero_total = int(
    zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    zero_total
)


if zero_total != 0:

    print(
        zero_counts[
            zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected source zero(s) found. "
        "Review against CDC before continuing."
    )


print(
    "PASS: Original OSQ_D contains no numeric zeros."
)


# ============================================================
# 6. GENUINE DECIMAL / FRACTIONAL CHECK
# ============================================================

#
# IMPORTANT:
#
# This prevents us from accidentally converting a real
# measurement/decimal variable to integer.
#
# We only use Int64 below AFTER proving all nonmissing
# OSQ_D values are mathematically whole numbers.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:

        fractional_columns[col] = count


if fractional_columns:

    print(
        "Fractional variables found:"
    )

    print(
        fractional_columns
    )

    raise ValueError(
        "STOP: Genuine decimal fields found. "
        "Do NOT convert those variables to Int64."
    )


print(
    "PASS: Genuine fractional variables = 0."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy()
)


if not np.all(
    np.isclose(
        seqn_values,
        np.round(seqn_values),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


seqn_missing = int(
    df["SEQN"]
    .isna()
    .sum()
)


seqn_duplicates = int(
    df["SEQN"]
    .duplicated()
    .sum()
)


seqn_min = df["SEQN"].min()
seqn_max = df["SEQN"].max()


print(
    "Missing:",
    seqn_missing
)

print(
    "Duplicates:",
    seqn_duplicates
)

print(
    "Minimum:",
    seqn_min
)

print(
    "Maximum:",
    seqn_max
)


if seqn_missing != 0:
    raise ValueError(
        "Unexpected missing SEQN."
    )


if seqn_duplicates != 0:
    raise ValueError(
        "Duplicate SEQN values found."
    )


if seqn_min != 31130:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn_min}"
    )


if seqn_max != 41473:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn_max}"
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. CDC KEY CATEGORICAL FREQUENCY CHECKS
# ============================================================

EXPECTED_FREQUENCIES = {

    "OSQ010A": {
        "counts": {
            1: 73,
            2: 4902,
            9: 4
        },
        "missing": 0
    },

    "OSQ010B": {
        "counts": {
            1: 472,
            2: 4502,
            9: 5
        },
        "missing": 0
    },

    "OSQ010C": {
        "counts": {
            1: 127,
            2: 4844,
            9: 8
        },
        "missing": 0
    },

    "OSQ080": {
        "counts": {
            1: 1052,
            2: 3918,
            9: 9
        },
        "missing": 0
    },

    "OSQ060": {
        "counts": {
            1: 254,
            2: 4714,
            9: 11
        },
        "missing": 0
    },

    "OSQ070": {
        "counts": {
            1: 182,
            2: 70,
            9: 1
        },
        "missing": 4726
    },

    "OSQ130": {
        "counts": {
            1: 245,
            2: 4690,
            9: 44
        },
        "missing": 0
    },

    "OSQ140U": {
        "counts": {
            1: 154,
            2: 83
        },
        "missing": 4742
    },

    "OSQ150": {
        "counts": {
            1: 422,
            2: 4319,
            9: 238
        },
        "missing": 0
    },

    "OSQ160A": {
        "counts": {
            1: 383
        },
        "missing": 4596
    },

    "OSQ160B": {
        "counts": {
            2: 55
        },
        "missing": 4924
    },

    "OSQ170": {
        "counts": {
            1: 242,
            2: 4568,
            9: 169
        },
        "missing": 0
    },

    "OSQ190": {
        "counts": {
            1: 4,
            2: 5,
            9: 1
        },
        "missing": 4969
    },

    "OSQ200": {
        "counts": {
            1: 106,
            2: 4596,
            9: 277
        },
        "missing": 0
    },

    "OSQ220": {
        "counts": {
            1: 4,
            2: 5,
            9: 1
        },
        "missing": 4969
    }
}


print(
    "\n--- CDC KEY FREQUENCY CHECKS ---"
)


for col, spec in EXPECTED_FREQUENCIES.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v
        in actual_counts.items()
    }


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_counts != spec["counts"]:

        raise ValueError(
            f"{col}: CDC frequency mismatch.\n"
            f"Expected: {spec['counts']}\n"
            f"Actual:   {actual_counts}"
        )


    if actual_missing != spec["missing"]:

        raise ValueError(
            f"{col}: missing count mismatch.\n"
            f"Expected: {spec['missing']:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col:<9} "
        f"valid={sum(actual_counts.values()):,} "
        f"missing={actual_missing:,}"
    )


print(
    "PASS: Key categorical frequencies match CDC."
)


# ============================================================
# 9. CDC CHECK - FRACTURE COUNTS
# ============================================================

print(
    "\n--- CDC FRACTURE COUNT CHECKS ---"
)


# OSQ020A
# Hip fractures:
# 1-3 = 73
# Missing = 4,906

s = df["OSQ020A"]


if not (
    int(
        s.between(
            1,
            3
        ).sum()
    ) == 73

    and

    int(
        s.isna()
        .sum()
    ) == 4906
):

    raise ValueError(
        "OSQ020A differs from CDC."
    )


print(
    "PASS: OSQ020A = 73 valid values "
    "in range 1-3."
)


# OSQ020B
# Wrist fractures:
# 1-7 = 471
# 9999 = 1
# Missing = 4,507

s = df["OSQ020B"]


if not (
    int(
        s.between(
            1,
            7
        ).sum()
    ) == 471

    and

    int(
        s.eq(9999)
        .sum()
    ) == 1

    and

    int(
        s.isna()
        .sum()
    ) == 4507
):

    raise ValueError(
        "OSQ020B differs from CDC."
    )


print(
    "PASS: OSQ020B matches CDC."
)


# OSQ020C
# Spine fractures:
# 1-10 = 125
# 9999 = 2
# Missing = 4,852

s = df["OSQ020C"]


if not (
    int(
        s.between(
            1,
            10
        ).sum()
    ) == 125

    and

    int(
        s.eq(9999)
        .sum()
    ) == 2

    and

    int(
        s.isna()
        .sum()
    ) == 4852
):

    raise ValueError(
        "OSQ020C differs from CDC."
    )


print(
    "PASS: OSQ020C matches CDC."
)


# ============================================================
# 10. CDC CHECK - OSQ140Q / OSQ140U
# ============================================================

#
# Duration of prednisone/cortisone use.
#
# OSQ140Q:
# 1-54 = 237
# 999  = 8
# Missing = 4,734
#
# OSQ140U:
# 1 Month = 154
# 2 Year  = 83
# Missing = 4,742
#
# IMPORTANT:
# Q and U are separate quantity/unit fields.
# DO NOT calculate a decimal duration.
#

s = df["OSQ140Q"]


if not (
    int(
        s.between(
            1,
            54
        ).sum()
    ) == 237

    and

    int(
        s.eq(999)
        .sum()
    ) == 8

    and

    int(
        s.isna()
        .sum()
    ) == 4734
):

    raise ValueError(
        "OSQ140Q differs from CDC."
    )


print(
    "\nPASS: OSQ140Q matches CDC."
)


# ============================================================
# 11. CDC CHECK - MOTHER'S HIP FRACTURE AGE
# ============================================================

#
# OSQ180:
# 3-97 = 232
# 999 Don't know = 10
# Missing = 4,737
#

s = df["OSQ180"]


if not (
    int(
        s.between(
            3,
            97
        ).sum()
    ) == 232

    and

    int(
        s.eq(999)
        .sum()
    ) == 10

    and

    int(
        s.isna()
        .sum()
    ) == 4737
):

    raise ValueError(
        "OSQ180 differs from CDC."
    )


print(
    "PASS: OSQ180 matches CDC."
)


# ============================================================
# 12. CDC CHECK - FATHER'S HIP FRACTURE AGE
# ============================================================

#
# OSQ210:
# 10-98 = 96
# 999 Don't know = 10
# Missing = 4,873
#

s = df["OSQ210"]


if not (
    int(
        s.between(
            10,
            98
        ).sum()
    ) == 96

    and

    int(
        s.eq(999)
        .sum()
    ) == 10

    and

    int(
        s.isna()
        .sum()
    ) == 4873
):

    raise ValueError(
        "OSQ210 differs from CDC."
    )


print(
    "PASS: OSQ210 matches CDC."
)


# ============================================================
# 13. EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# OSQ_D HAS NO NUMERIC CORRECTIONS.
#
# No:
#
# - tiny -> zero correction
# - rounding
# - recalculation
# - decimal conversion
#
# Therefore expected_df is exactly the original XPT.
#

expected_df = source_df.copy()


# ============================================================
# 14. CLEAN WHOLE-NUMBER QUESTIONNAIRE FORMATTING
# ============================================================

#
# We have already proven:
#
# - all 118 fields are numeric in the source
# - every nonmissing value is mathematically whole
# - there are no genuine decimal variables
#
# Therefore Int64 is safe for this file.
#
# Examples:
#
# SEQN
# 31130.0 -> 31130
#
# OSQ010A
# 1.0 -> 1
#
# OSD030AA
# 85.0 -> 85
#
# OSQ020B
# 9999.0 -> 9999
#
# Missing
# NaN -> blank
#

for col in df.columns:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All whole-number OSQ_D fields "
    "converted to clean integer formatting."
)


# ============================================================
# 15. PRE-EXPORT VALUE FIDELITY
# ============================================================

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: No OSQ_D numeric values changed."
)


# ============================================================
# 16. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# No float_format
# No round()
# No recalculation
#
# Missing SAS values become blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 17. RE-READ CSV WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 18. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 19. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- XPT -> CSV NUMERIC FIDELITY ---"
)


numeric_failures = []
total_numeric_differences = 0


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += differences


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Total numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 118 CSV columns numerically "
    "match the original XPT exactly."
)


# ============================================================
# 20. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    diff = int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    missing_mismatches += diff


    if diff > 0:

        missing_failures.append(
            (
                col,
                diff
            )
        )


print(
    "\nMissing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(missing_failures)
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 21. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(
        reader
    )


if len(raw_rows) != 4979:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 22. STRICT RAW CSV FORMATTING CHECK
# ============================================================

#
# Because every source value is a proven whole-number field,
# expected CSV representation is:
#
# missing -> ""
#
# otherwise:
# str(int(source_value))
#
# Examples:
#
# 31130.0 -> "31130"
# 1.0     -> "1"
# 85.0    -> "85"
# 9999.0  -> "9999"
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in expected_df.columns:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "OSQ_D raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV formatting matches "
    "expected questionnaire representation."
)


# ============================================================
# 23. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        "Examples:",
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 24. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 25. FINAL TINY-ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 26. FINAL CDC FREQUENCY CHECK AFTER EXPORT
# ============================================================

print(
    "\n--- FINAL CSV CDC FREQUENCY CHECK ---"
)


for col, spec in EXPECTED_FREQUENCIES.items():

    actual_counts = (
        csv_df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v
        in actual_counts.items()
    }


    actual_missing = int(
        csv_df[col]
        .isna()
        .sum()
    )


    if actual_counts != spec["counts"]:

        raise ValueError(
            f"{col}: CSV frequencies changed."
        )


    if actual_missing != spec["missing"]:

        raise ValueError(
            f"{col}: CSV missing count changed."
        )


print(
    "PASS: Final CSV key frequencies still match CDC."
)


# ============================================================
# 27. FINAL FRACTIONAL CHECK
# ============================================================

#
# There should be no decimal values in OSQ_D.
#

fractional_csv_columns = {}


for col in csv_df.columns:

    values = (
        csv_df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:

        fractional_csv_columns[col] = count


if fractional_csv_columns:

    raise ValueError(
        "Unexpected decimal values in final CSV:\n"
        + str(fractional_csv_columns)
    )


print(
    "PASS: Final CSV genuine decimal variables = 0."
)


# ============================================================
# 28. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("OSQ_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Tiny artifacts in original XPT:",
    tiny_total
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "True source zeros:",
    zero_total
)

print(
    "Genuine decimal variables:",
    0
)

print(
    "SEQN/code/count/age formatting:",
    "integer"
)

print(
    "XPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Values ending .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)

print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "OSQ_D_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2005-2006 OSQ_D XPT -> CSV
Rows: 4,979
Columns: 118
PASS: OSQ_D structure = 4,979 rows x 118 columns.

--- TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: OSQ_D contains no tiny-value artifacts.

True numeric zeros: 0
PASS: Original OSQ_D contains no numeric zeros.

--- FRACTIONAL VALUE CHECK ---
PASS: Genuine fractional variables = 0.

--- SEQN CHECK ---
Missing: 0
Duplicates: 0
Minimum: 31130.0
Maximum: 41473.0
PASS: SEQN validated.

--- CDC KEY FREQUENCY CHECKS ---
PASS: OSQ010A   valid=4,979 missing=0
PASS: OSQ010B   valid=4,979 missing=0
PASS: OSQ010C   valid=4,979 missing=0
PASS: OSQ080    valid=4,979 missing=0
PASS: OSQ060    valid=4,979 missing=0
PASS: OSQ070    valid=253 missing=4,726
PASS: OSQ130    valid=4,979 missing=0
PASS: OSQ140U   valid=237 missing=4,742
PASS: OSQ150    valid=4,979 missing=0
PASS: OSQ160A   valid=383 missing=4,596
PASS: OSQ160B   valid=55 missing=4,924
PASS: OSQ170    valid=4,979 missing=0
PASS: OSQ190    valid=10 missing=4,969
PASS: OSQ2